# Evaluation

Produces the report's headline table: three systems scored on the same
photographs.

| Column | What it is |
|---|---|
| Field-trained | The model AgroMet ships, trained on cassava field imagery |
| Lab-trained | The same architecture trained on PlantVillage |
| Kindwise | The commercial API the app calls when online |

The test set is the Ghanaian field photographs, labelled with an extension
officer present. None of the three has seen any of them.

Run `ml/scripts/benchmark_kindwise.py` first to produce the Kindwise column.

In [ ]:
import json, pathlib
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix

IMAGE_SIZE = 224
CLASS_IDS = ["cbb", "cbsd", "cgm", "cmd", "healthy"]

FIELD_DIR = pathlib.Path("ml/data/field")      # photographs, one folder per class
LABELS_CSV = FIELD_DIR / "labels.csv"          # filename, class_id, labelled_by, confidence

## Load the field set

`labels.csv` carries who labelled each photograph and how sure they were.
Disagreements are kept rather than resolved away: a photograph two officers read
differently is a real fact about how hard the task is, and dropping it flatters
every model in the table.

In [ ]:
field = pd.read_csv(LABELS_CSV)
print(len(field), "photographs")
print(field["class_id"].value_counts())

if "labelled_by" in field:
    disputed = field[field.duplicated("filename", keep=False)]
    print(f"{disputed['filename'].nunique()} photographs with more than one label")

In [ ]:
def load_image(path):
    image = tf.io.decode_jpeg(tf.io.read_file(str(path)), channels=3)
    image = tf.image.resize(image, [IMAGE_SIZE, IMAGE_SIZE])
    return tf.expand_dims(image, 0)  # 0-255, matching the app

def score(model_path, frame):
    model = tf.keras.models.load_model(model_path)
    preds = [model.predict(load_image(FIELD_DIR / row.filename), verbose=0).argmax() for row in frame.itertuples()]
    return np.array(preds)

y_true = field["class_id"].map(CLASS_IDS.index).values

In [ ]:
for name, path in [("field-trained", "ml/models/cassava.keras"), ("lab-trained", "ml/models/plantvillage.keras")]:
    y_pred = score(path, field)
    print(f"\n===== {name} =====")
    print(classification_report(y_true, y_pred, target_names=CLASS_IDS, digits=3))
    print(confusion_matrix(y_true, y_pred))

## Kindwise

Scored from the JSON that `benchmark_kindwise.py` wrote. Its labels are free
text disease names, not our five class ids, so the mapping is explicit and
recorded rather than fuzzy-matched: a comparison that quietly resolves ambiguous
names in the API's favour is not a comparison.

In [ ]:
kindwise = json.loads(pathlib.Path("ml/data/kindwise_results.json").read_text())

# Filled in by hand after reading the distinct names the API actually returned.
KINDWISE_TO_CLASS = {
    # "Cassava mosaic virus": "cmd",
    # "Cassava brown streak virus": "cbsd",
    # "Cassava bacterial blight": "cbb",
}

names = sorted({entry["disease"] for entry in kindwise if entry.get("disease")})
print("Map these before scoring:")
for name in names:
    print(f"  {name!r}: {KINDWISE_TO_CLASS.get(name, '???')}")

## Also record, because they justify the architecture

- On-device inference latency on a real mid-range phone
- Bundled model size
- What fraction of photographs each system declined to answer at all

A model that answers 60% of the time at high precision may serve a farmer better
than one that always answers and is wrong a third of the time. The comparison
table should show both the accuracy and the coverage it was achieved at.